# Pharmaceutical Inventory & Demand Planning Analytics
### End-to-End Case Study | Python + Excel + Power BI

**Author:** Hossam Badawy — Licensed Pharmacist | Supply Chain & Inventory Analyst  
**Tools:** Python · Excel · Power BI (PL-300)  
**Dataset:** Pharmaceutical Sales Data 2014–2019 (12 drugs, ATC classification)  

---

## Project Overview

This notebook covers the Python component of a full pharmaceutical inventory analytics project:

1. **Data Loading & Exploration** — understanding the dataset structure and drug categories
2. **Outlier Detection & Treatment** — Winsorization of anomalous data points
3. **Demand Forecasting** — Holt-Winters Exponential Smoothing per drug category
4. **Forecast Accuracy** — MAPE calculation on 2019 test period
5. **2020 Demand Plan** — 12-month forecast with 95% confidence intervals

> **ABC-XYZ Classification and Safety Stock / ROP calculations** are in the accompanying Excel file.  
> **Power BI Dashboard** visualizes all outputs from both Python and Excel.

---

## Step 1 — Install & Import Libraries

In [ ]:
!pip install statsmodels --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

## Step 2 — Load & Explore the Data

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the dataset
df = pd.read_csv('working_data.csv')
df = df.dropna(subset=['date'])
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df = df.sort_values('date').reset_index(drop=True)

# Drug columns
drug_cols = [c for c in df.columns if c not in
             ['date', 'year', 'month', 'total sales', 'status']]

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["date"].min().strftime("%b %Y")} to {df["date"].max().strftime("%b %Y")}')
print(f'Drug categories: {len(drug_cols)}')
print(f'\nDrug columns:')
for i, drug in enumerate(drug_cols, 1):
    print(f'  {i:2}. {drug}')

In [ ]:

fig, axes = plt.subplots(4, 3, figsize=(18, 14))
fig.suptitle('Monthly Sales by Drug Category (2014-2019)',
             fontsize=16, fontweight='bold', y=1.02)

colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D',
          '#3B1F2B', '#44BBA4', '#E94F37', '#393E41',
          '#F5A623', '#7B2D8B', '#00A86B', '#FF6B6B']

for idx, (drug, ax) in enumerate(zip(drug_cols, axes.flatten())):
    ax.plot(df['date'], df[drug], color=colors[idx], linewidth=1.5)
    ax.set_title(drug.split('(')[-1].replace(')', '').strip(),
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3)
    ax.axvspan(pd.Timestamp('2019-01-01'), df['date'].max(),
               alpha=0.1, color='orange', label='Test Period')

plt.tight_layout()
plt.savefig('drug_sales_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as drug_sales_overview.png')

## Step 3 — Outlier Detection & Winsorization

During exploration, October 2019 showed a **70-85% drop** across all drug categories —  
statistically impossible as a real demand event. This is treated as a data quality issue.

**Method: Winsorization** — values below (Mean − 2×StdDev) are capped at the lower bound.  
This preserves the data record while removing the distorting effect of the anomaly.

In [ ]:
# Investigatin October 2019 anomaly
oct_2019 = df[df['date'] == '2019-10-31']
print('October 2019 vs Historical October Average (2014-2018):')
print(f'{"Drug":<35} {"Oct-2019":>10} {"Hist Avg":>10} {"Change":>10}')
print('-' * 70)

for drug in drug_cols:
    oct_val = oct_2019[drug].values[0]
    hist_avg = df[(df['month'] == 10) & (df['year'] <= 2018)][drug].mean()
    change = ((oct_val - hist_avg) / hist_avg) * 100
    flag = ' ← ANOMALY' if change < -50 else ''
    print(f'{drug[:34]:<35} {oct_val:>10.1f} {hist_avg:>10.1f} {change:>+9.0f}%{flag}')

In [ ]:
# Applying Winsorization (replace October 2019 with historical October avg)
oct_idx = df[df['date'] == '2019-10-31'].index[0]

for drug in drug_cols:
    hist_oct_avg = df[(df['month'] == 10) & (df['year'] <= 2018)][drug].mean()
    df.loc[oct_idx, drug] = round(hist_oct_avg, 2)

print('Winsorization applied — October 2019 replaced with historical averages')
print('Data is now clean and ready for forecasting')

## Step 4 — Train/Test Split

| Period | Dates | Purpose |
|--------|-------|--------|
| **Train** | Jan 2014 — Dec 2018 | Model learns the pattern |
| **Test** | Jan 2019 — Oct 2019 | Forecast accuracy measurement (MAPE) |
| **Forecast** | Jan 2020 — Dec 2020 | Final 12-month demand plan |

In [ ]:
train = df[df['year'] <= 2018].copy()
test  = df[df['year'] == 2019].copy()
forecast_dates = pd.date_range(start='2020-01-31', periods=12, freq='ME')

print(f'Training period: {len(train)} months ({train["date"].min().strftime("%b %Y")} - {train["date"].max().strftime("%b %Y")})')
print(f'Test period:     {len(test)} months ({test["date"].min().strftime("%b %Y")} - {test["date"].max().strftime("%b %Y")})')
print(f'Forecast period: 12 months (Jan 2020 - Dec 2020)')

## Step 5 — Model Selection

Different drugs require different forecasting models based on their demand pattern:

| Model | When to Use | Drugs |
|-------|------------|-------|
| **Holt-Winters (trend + seasonal)** | Clear trend and seasonality | Most drugs |
| **Holt-Winters (seasonal only)** | Seasonality, no clear trend | Hypnotics, Antimalarial, Oseltamivir |

> Oseltamivir and Antimalarial are **event-driven** (flu season, Hajj calendar) —  
> high MAPE is expected and these drugs require manual planning overlay.

In [ ]:
# Model configuration per drug
model_config = {
    'M01AB (Diclofenac)'   : {'trend': 'add', 'seasonal': 'add'},
    'M01AE (Ibuprofen)'    : {'trend': 'add', 'seasonal': 'add'},
    'N02BA (Aspirin )'     : {'trend': 'add', 'seasonal': 'add'},
    'N02BE (Paracetamol )' : {'trend': 'add', 'seasonal': 'add'},
    'N05B (Anxiolytics )'  : {'trend': 'add', 'seasonal': 'add'},
    'N05C (Hypnotics )'    : {'trend': None,  'seasonal': 'add'},
    'R03 (Inhalers)'       : {'trend': 'add', 'seasonal': 'add'},
    'J05AH02 (Oseltamivir)': {'trend': None,  'seasonal': 'add'},
    'A07CA (ORS)'          : {'trend': 'add', 'seasonal': 'add'},
    'A11CC05 (Vitamin D)'  : {'trend': 'add', 'seasonal': 'add'},
    'P01BA01 (Antimalarial)':{'trend': None,  'seasonal': 'add'},
    'R06 (Antihistamines)' : {'trend': 'add', 'seasonal': 'add'},
}
print('Model configuration set for all 12 drugs')

## Step 6 — Forecasting & MAPE Calculation

In [ ]:
results      = []
mape_summary = []

for drug in drug_cols:
    train_vals = train[drug].values.astype(float)
    test_vals  = test[drug].values.astype(float)
    cfg        = model_config[drug]

    # Fit on training data
    try:
        model = ExponentialSmoothing(
            train_vals, trend=cfg['trend'], seasonal=cfg['seasonal'],
            seasonal_periods=12, initialization_method='estimated'
        ).fit(optimized=True)
    except:
        model = ExponentialSmoothing(
            train_vals, trend=None, seasonal='add',
            seasonal_periods=12, initialization_method='estimated'
        ).fit(optimized=True)

    # Test period forecast & MAPE
    test_forecast = model.forecast(len(test_vals))
    mape = np.mean(np.abs((test_vals - test_forecast) / test_vals) * 100)
    quality = 'Good' if mape < 20 else ('Acceptable' if mape < 35 else 'Review — Manual Planning Required')
    mape_summary.append({'Drug': drug, 'MAPE_%': round(mape, 1), 'Quality': quality})

    # Refit on all data then forecast 2020
    all_vals = df[drug].values.astype(float)
    try:
        model_full = ExponentialSmoothing(
            all_vals, trend=cfg['trend'], seasonal=cfg['seasonal'],
            seasonal_periods=12, initialization_method='estimated'
        ).fit(optimized=True)
    except:
        model_full = ExponentialSmoothing(
            all_vals, trend=None, seasonal='add',
            seasonal_periods=12, initialization_method='estimated'
        ).fit(optimized=True)

    forecast_2020 = np.maximum(model_full.forecast(12), 0)
    residual_std  = np.std(all_vals - model_full.fittedvalues)
    upper = forecast_2020 + 1.96 * residual_std
    lower = np.maximum(forecast_2020 - 1.96 * residual_std, 0)

    # Training rows
    for i, row in train.iterrows():
        results.append({'Date': row['date'].strftime('%Y-%m-%d'), 'Drug': drug,
                        'Actual': round(row[drug], 2),
                        'Forecasted': round(model.fittedvalues[train.index.get_loc(i)], 2),
                        'Error': None, 'Abs_Error_Pct': None,
                        'Upper': None, 'Lower': None, 'Period': 'Train', 'MAPE_%': None})

    # Test rows
    for i, (_, row) in enumerate(test.iterrows()):
        err     = round(row[drug] - test_forecast[i], 2)
        abs_pct = round(abs(err / row[drug]) * 100, 1) if row[drug] != 0 else None
        results.append({'Date': row['date'].strftime('%Y-%m-%d'), 'Drug': drug,
                        'Actual': round(row[drug], 2), 'Forecasted': round(test_forecast[i], 2),
                        'Error': err, 'Abs_Error_Pct': abs_pct,
                        'Upper': None, 'Lower': None, 'Period': 'Test', 'MAPE_%': round(mape, 1)})

    # 2020 forecast rows
    for i, fdate in enumerate(forecast_dates):
        results.append({'Date': fdate.strftime('%Y-%m-%d'), 'Drug': drug,
                        'Actual': None, 'Forecasted': round(forecast_2020[i], 2),
                        'Error': None, 'Abs_Error_Pct': None,
                        'Upper': round(upper[i], 2), 'Lower': round(lower[i], 2),
                        'Period': 'Forecast', 'MAPE_%': round(mape, 1)})

results_df = pd.DataFrame(results)
mape_df    = pd.DataFrame(mape_summary)
print(f'Forecasting complete — {len(results_df):,} rows generated')

## Step 7 — MAPE Results

In [ ]:
# Displaying MAPE summary with color coding
print('=' * 65)
print(f'{"FORECAST ACCURACY SUMMARY (Test Period: Jan-Oct 2019)":^65}')
print('=' * 65)
print(f'{"Drug":<35} {"MAPE":>8}  {"Quality"}')
print('-' * 65)

for _, row in mape_df.sort_values('MAPE_%').iterrows():
    icon = '✅' if row['MAPE_%'] < 20 else ('⚠️' if row['MAPE_%'] < 35 else '❌')
    print(f"{row['Drug'][:34]:<35} {row['MAPE_%']:>7.1f}%  {icon} {row['Quality']}")

print('-' * 65)
good = len(mape_df[mape_df['MAPE_%'] < 20])
acceptable = len(mape_df[(mape_df['MAPE_%'] >= 20) & (mape_df['MAPE_%'] < 35)])
review = len(mape_df[mape_df['MAPE_%'] >= 35])
print(f'\n✅ Good (MAPE < 20%):        {good} drugs')
print(f'⚠️  Acceptable (20-35%):      {acceptable} drugs')
print(f'❌ Manual Planning Required: {review} drugs (event-driven demand)')

## Step 8 — Visualize Forecast vs Actual

In [ ]:
# Plot forecast vs actual for 4 representative drugs
selected_drugs = [
    'N02BE (Paracetamol )',   # AX — stable high volume
    'R06 (Antihistamines)',   # CY — seasonal
    'A07CA (ORS)',            # BY — summer peak
    'J05AH02 (Oseltamivir)',  # CZ — event driven
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Forecast vs Actual — Representative Drug Categories',
             fontsize=14, fontweight='bold')

for ax, drug in zip(axes.flatten(), selected_drugs):
    drug_data = results_df[results_df['Drug'] == drug].copy()
    drug_data['Date'] = pd.to_datetime(drug_data['Date'])

    train_d    = drug_data[drug_data['Period'] == 'Train']
    test_d     = drug_data[drug_data['Period'] == 'Test']
    forecast_d = drug_data[drug_data['Period'] == 'Forecast']
    mape_val   = mape_df[mape_df['Drug'] == drug]['MAPE_%'].values[0]

    ax.plot(train_d['Date'], train_d['Actual'],
            color='#2E86AB', linewidth=1.5, label='Actual (Train)')
    ax.plot(test_d['Date'], test_d['Actual'],
            color='#2E86AB', linewidth=2.5, label='Actual (Test)')
    ax.plot(test_d['Date'], test_d['Forecasted'],
            color='#F18F01', linewidth=2, linestyle='--', label='Forecasted')
    ax.plot(forecast_d['Date'], forecast_d['Forecasted'],
            color='#F18F01', linewidth=2, label='2020 Forecast')
    ax.fill_between(forecast_d['Date'],
                    forecast_d['Lower'], forecast_d['Upper'],
                    alpha=0.2, color='#F18F01', label='95% CI')
    ax.axvline(pd.Timestamp('2019-01-01'), color='gray',
               linestyle=':', linewidth=1, alpha=0.7)
    ax.axvline(pd.Timestamp('2020-01-31'), color='red',
               linestyle=':', linewidth=1.5, alpha=0.8, label='Forecast Start')

    short_name = drug.split('(')[-1].replace(')', '').strip()
    ax.set_title(f'{short_name} | MAPE: {mape_val}%', fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig('forecast_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as forecast_visualization.png')

## Step 9 — Export Results for Power BI

In [ ]:
# Export both output files
results_df.to_csv('forecast_results.csv', index=False)
mape_df.to_csv('mape_summary.csv', index=False)

print('Files exported successfully:')
print(f'  forecast_results.csv — {len(results_df):,} rows ({results_df["Drug"].nunique()} drugs × train/test/forecast periods)')
print(f'  mape_summary.csv     — {len(mape_df)} rows (one per drug)')
print('\nThese files are imported directly into Power BI for dashboard visualization.')

# Download files
files.download('forecast_results.csv')
files.download('mape_summary.csv')